# Running GPFA to reproduce figure 3

The GPFA part of the analysis is done using the [`Elephant`](https://elephant.readthedocs.io/) library.
The code is based on the tutorial available at [Tutorial: GPFA (Gaussian Process Factor Analysis)](https://elephant.readthedocs.io/en/latest/tutorials/gpfa.html).

## 1. The Idea of This Notebook

This tutorial illustrates the usage of the `gpfa.GPFA()` class implemented in elephant, through its applications to synthetic spike train data, of which the ground truth low-dimensional structure is known.

Different decoding pipelines to decode the heading angle from the latent variables are demonstrated&mdash;with data leakage and without data leakage.

### 1.1. Generation of synthetic spike trains

A set of spike trains are generated as follows.
First, a time series of either a 2-dimensional harmonic oscillator (Section 2) is projected into a high-dimensional space (as high dimension as the desired number of parallel spike trains) via a random projection.
Then the resulting high-dimensional time series serves as time-dependent rates for an inhomogeneous multivariate Poisson process.
Finally, multiple realizations of this Poisson process, which mimic spike trains from multiple trials, serve as input data to the GPFA.

Below are the functions used for spike train generation:

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import KNeighborsRegressor
import pandas as pd
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import KFold
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from tqdm.auto import tqdm, trange
from scipy.stats import wilcoxon

import quantities as pq
import neo
from elephant.spike_train_generation import inhomogeneous_poisson_process
from elephant.gpfa import GPFA
from gpfa_core import circular_rmse
from io_utils import panel_results_path, save_gpfa_panel_results

plt.style.use("main.mplstyle")

In [ ]:
def integrated_oscillator(
    dt, num_steps, x0=0, y0=1, angular_frequency=2 * np.pi * 1e-3
):
    """
    Parameters
    ----------
    dt : float
        Integration time step in ms.
    num_steps : int
        Number of integration steps -> max_time = dt*(num_steps-1).
    x0, y0 : float
        Initial values in three dimensional space.
    angular_frequency : float
        Angular frequency in 1/ms.

    Returns
    -------
    t : (num_steps) np.ndarray
        Array of timepoints
    (2, num_steps) np.ndarray
        Integrated two-dimensional trajectory (x, y, z) of the harmonic oscillator
    """

    assert isinstance(num_steps, int), "num_steps has to be integer"
    t = dt * np.arange(num_steps)
    x = x0 * np.cos(angular_frequency * t) + y0 * np.sin(angular_frequency * t)
    y = -x0 * np.sin(angular_frequency * t) + y0 * np.cos(angular_frequency * t)
    return t, np.array((x, y))


def random_projection(data, embedding_dimension, loc=0, scale=None):
    """
    Parameters
    ----------
    data : np.ndarray
        Data to embed, shape=(M, N)
    embedding_dimension : int
        Embedding dimension, dimensionality of the space to project to.
    loc : float or array_like of floats
        Mean ("centre") of the distribution.
    scale : float or array_like of floats
        Standard deviation (spread or "width") of the distribution.

    Returns
    -------
    np.ndarray
       Random (normal) projection of input data, shape=(dim, N)

    See Also
    --------
    np.random.normal()

    """
    if scale is None:
        scale = 1 / np.sqrt(data.shape[0])
    projection_matrix = np.random.normal(
        loc, scale, (embedding_dimension, data.shape[0])
    )
    return projection_matrix, projection_matrix @ data


def generate_spiketrains(instantaneous_rates, num_trials, timestep):
    """
    Parameters
    ----------
    instantaneous_rates : np.ndarray
        Array containing time series.
    timestep :
        Sample period.
    num_steps : int
        Number of timesteps -> max_time = timestep*(num_steps-1).

    Returns
    -------
    spiketrains : list of neo.SpikeTrains
        List containing spiketrains of inhomogeneous Poisson
        processes based on given instantaneous rates.

    """

    spiketrains = []
    for _ in range(num_trials):
        spiketrains_per_trial = []
        for inst_rate in instantaneous_rates:
            anasig_inst_rate = neo.AnalogSignal(
                inst_rate, sampling_rate=1 / timestep, units=pq.Hz
            )
            spiketrains_per_trial.append(
                inhomogeneous_poisson_process(anasig_inst_rate)
            )
        spiketrains.append(spiketrains_per_trial)

    return spiketrains

## 2. Harmonic Oscillator Example

In this first example we apply the GPFA to spike train data derived from dynamics of a harmonic oscillator defined in a 2-dimensional latent variable space. The aim is to extract these 2-dimensional latent dynamics from the spike train data.

### 2.1. Generate synthetic spike train data
Here we generate 50-dimensional synthetic spike train data based on a trajectory of a 2-dimensional harmonic oscillator, as described in Section 1.1.

In [ ]:
# set parameters for the integration of the harmonic oscillator
timestep = 1 * pq.ms
trial_duration = 2 * pq.s
num_steps = int((trial_duration.rescale("ms") / timestep).magnitude)

# set parameters for spike train generation
MAX_RATE = 150 * pq.Hz
np.random.seed(
    42
)  # for visualization purposes, we want to get identical spike trains at any run

# specify data size
num_trials = 20
num_spiketrains = 50

# generate a low-dimensional trajectory
times_oscillator, oscillator_trajectory_2dim = integrated_oscillator(
    timestep.magnitude, num_steps=num_steps, x0=0, y0=1
)
times_oscillator = (times_oscillator * timestep.units).rescale("s")

# random projection to high-dimensional space
projection_matrix, oscillator_trajectory_Ndim = random_projection(
    oscillator_trajectory_2dim, embedding_dimension=num_spiketrains
)

spiketrains_oscillator = []
data = oscillator_trajectory_2dim
scale = 1 / np.sqrt(data.shape[0])
embedding_dimension = num_spiketrains

oscillator_trajectory_Ndim = projection_matrix @ oscillator_trajectory_2dim
# convert to instantaneous rate for Poisson process
normed_traj = oscillator_trajectory_Ndim / oscillator_trajectory_Ndim.max()
instantaneous_rates_oscillator = (
    np.power(MAX_RATE.magnitude, normed_traj) + 10
)  # 10 Hz baseline

for i in range(num_trials):
    # generate spike trains
    spiketrains_oscillator_tmp = generate_spiketrains(
        instantaneous_rates_oscillator, 1, timestep
    )

    spiketrains_oscillator.extend(spiketrains_oscillator_tmp)

In [ ]:
# held-out data for testing, large number of trials for evaluating generalization error
spiketrains_oscillator_test = []

for i in trange(1000):
    # generate spike trains
    spiketrains_oscillator_tmp = generate_spiketrains(
        instantaneous_rates_oscillator, 1, timestep
    )
    spiketrains_oscillator_test.extend(spiketrains_oscillator_tmp)

## Inverse transform the firing rate data

In [ ]:
latent_degree = np.rad2deg(
    np.arctan2(oscillator_trajectory_2dim[1], oscillator_trajectory_2dim[0])
)

# use the base projection matrix on the last trial
tmp = np.linalg.pinv(projection_matrix) @ oscillator_trajectory_Ndim

fig, ax = plt.subplots(figsize=(3, 2), layout="constrained")

ax.scatter(*tmp, c=latent_degree, cmap="hsv", s=5)
ax.set_xlabel("Dim 1")
ax.set_ylabel("Dim 2")
ax.set_aspect(1)

# show colorbar
sm = plt.cm.ScalarMappable(cmap="hsv", norm=plt.Normalize(vmin=-180, vmax=180))
sm._A = []
cbar = fig.colorbar(sm, ax=ax)
cbar.ax.yaxis.set_ticks_position("left")
# cbar.ax.yaxis.set_label_position("left")
cbar.set_label("Latent angle (degrees)")
# set y-axis ticks
ax.set_yticks([-1, 0, 1])

[x.patch.set_visible(False) for x in (fig, ax)]

# fig.savefig("spiketrains_oscillator.svg", transparent=True)

Thus, we have generated 500-dimensional spike train data, derived from 2-dimensional latent dynamics, i.e., two cycles of circular rotation.

### 2.2. Apply GPFA to the generated data
Now we try to extract the original latent dynamics from the generated spike train data, by means of GPFA.

We first initialize an instance of the `gpfa.GPFA()` class.
One can specify some parameters for model fitting at this timing.
Here we set the size of the bin for spike train binning to 20 ms, and the dimensionality of latent variables to 2.

In [ ]:
# specify fitting parameters
bin_size = 20 * pq.ms
latent_dimensionality = 2

gpfa_2dim = GPFA(bin_size=bin_size, x_dim=latent_dimensionality)

In [ ]:
# here we just reuse the existing instance of the GPFA() class as we use the same fitting parameters as before
trajectories_all = gpfa_2dim.fit_transform(spiketrains_oscillator)
# transform the test data for generalization error evaluation
X_gen = gpfa_2dim.transform(spiketrains_oscillator_test)

# here we assume the array shapes are consistent across trials
trajectories_all = np.stack(trajectories_all)
X_gen = np.stack(X_gen)

Let's see how the extracted trajectories look like.

In [ ]:
linewidth_single_trial = 0.5
color_single_trial = "gray"
alpha_single_trial = 0.5

linewidth_trial_average = 2
color_trial_average = "black"

In [ ]:
fig, ax = plt.subplots(figsize=(3, 3))

for i, single_trial_trajectory in enumerate(trajectories_all):
    if i == 0:
        ax.plot(
            single_trial_trajectory[0],
            single_trial_trajectory[1],
            "-",
            lw=linewidth_single_trial,
            c=color_single_trial,
            alpha=alpha_single_trial,
            label="Single trial trajectory",
        )
    ax.plot(
        single_trial_trajectory[0],
        single_trial_trajectory[1],
        "-",
        lw=linewidth_single_trial,
        c=color_single_trial,
        alpha=alpha_single_trial,
    )

average_trajectory = np.mean(trajectories_all, axis=0)
ax.plot(
    average_trajectory[0],
    average_trajectory[1],
    "-",
    lw=linewidth_trial_average,
    c=color_trial_average,
    label="Trial averaged trajectory",
)
ax.set(ylim=(-1.3, 0.9), aspect=1, xlabel="Dim 1", ylabel="Dim 2")
ax.legend(loc="lower center")
[x.patch.set_visible(False) for x in (fig, ax)]

fig.savefig("latent_dynamics.svg", transparent=True)

In [ ]:
bin_size_int = int(bin_size.magnitude)
binned_trajectory = oscillator_trajectory_2dim.reshape(
    2, num_steps // bin_size_int, bin_size_int
).mean(-1)

y_gen = np.tile(binned_trajectory, len(X_gen)).T

We obtain almost the same latent dynamics, but single trial trajectories are slightly modified owing to an increased amount of the data used for fitting.

## 3. Decoding

### 3.1 with data leakage

In [ ]:
# multioutput regressor is a wrapper that allows to use single-output regressors to predict multiple outputs
# this is more appropriate for circular data
regressor = MultiOutputRegressor(KNeighborsRegressor(n_neighbors=5))

In [ ]:
cv = KFold(n_splits=5)

res = []
PLOT_FLAG: bool = True
for train_idx, test_idx in tqdm(cv.split(trajectories_all)):
    y_train = np.tile(binned_trajectory, len(train_idx)).T
    y_test = np.tile(binned_trajectory, len(test_idx)).T
    X_train = np.hstack(trajectories_all[train_idx]).T
    X_test = np.hstack(trajectories_all[test_idx]).T
    regressor.fit(X_train, y_train)
    y_pred = regressor.predict(X_test)

    if PLOT_FLAG:
        PLOT_FLAG = False
        # keep the fold the panel shows; y_pred is reused for gen. err. below
        panel_fold = (y_test, y_pred, test_idx)
        fig, ax = plt.subplots()
        ax.scatter(
            np.rad2deg(np.arctan2(y_pred[:, 1], y_pred[:, 0])),
            np.rad2deg(np.arctan2(y_test[:, 1], y_test[:, 0])),
            c="black",
            s=5,
            edgecolor="none",
            alpha=0.8,
        )
        ax.set(aspect="equal", xlabel="Predicted angle", ylabel="True angle")
        # ax.plot([-180, 180], [-180, 180], "--", c="gray", lw=1)

        [x.patch.set_visible(False) for x in (fig, ax)]
        fig.savefig("gpfa_regression.svg", transparent=True)

    # RMSE in the leakage case
    rmse = circular_rmse(y_pred, y_test)

    # RMSE of generalization error
    y_pred = regressor.predict(np.hstack(X_gen).T)
    rmse_test = circular_rmse(y_pred, y_gen)

    res.append((rmse, rmse_test))

In [ ]:
# %% persist what panels k and l show
# The source-data export reads this file rather than re-simulating, so the
# workbook and the figure can never drift apart.
save_gpfa_panel_results(
    panel_results_path("gpfa.ipynb"), trajectories_all, *panel_fold
)


In [ ]:
df = pd.DataFrame(res, columns=["leakage", "generalization"])
sns.stripplot(data=df, palette="Set2")

# Run scripts on computing cluster

In [ ]:
df = pd.read_hdf("gpfa_simulation_results.h5")
df.set_index("run_id", inplace=True)
df

In [ ]:
s1, p1 = wilcoxon(df["leaky_rmse"].values, df["gen_err_rmse"].values)
s2, p2 = wilcoxon(df["leaky_rmse"].values, df["no_leak_rmse"].values)
s3, p3 = wilcoxon(df["no_leak_rmse"].values, df["gen_err_rmse"].values)
p1, p2, p3

In [ ]:
df_tmp = df.melt(
    value_vars=df.columns,
    var_name="type",
    value_name="score",
)
df_tmp["type"] = df_tmp["type"].map({
    "leaky_rmse": "Leaky CV",
    "no_leak_rmse": "CV",
    "gen_err_rmse": "Gen error",
    "gen_err_rmse2": "gen error2",
})

In [ ]:
fig, ax = plt.subplots(figsize=(2, 2))
sns.boxplot(
    ax=ax,
    data=df_tmp.query("type != 'gen error2'"),
    x="type",
    y="score",
    color="k",
    fill=None,
    width=0.4,
    order=["Leaky CV", "Gen error", "CV"],
)
# set x-axis tick font size
ax.tick_params(axis='x', labelsize=8)
ax.set(xlabel="", ylabel="RMSE (degrees)")
# set x-axis label font size
[x.patch.set_visible(False) for x in (fig, ax)]


# --- significance brackets ---
# Scaled to the spread of the boxes rather than to fixed offsets, since the box
# plot spans the full across-run distribution instead of a zoomed few degrees.
y_max = df_tmp["score"].max()
span = y_max - df_tmp["score"].min()
h = 0.02 * span  # height of the bracket ends
lvl1 = y_max + 0.06 * span  # pairwise brackets between neighbours
lvl2 = lvl1 + 0.10 * span  # outer bracket spanning all three


def annotate(x1, x2, y, label):
    ax.plot([x1, x1, x2, x2], [y, y + h, y + h, y], lw=1.5, c="k")
    ax.text(
        (x1 + x2) * 0.5, y + h * 1.4, label, ha="center", va="bottom", fontsize=7
    )


annotate(0.1, 0.9, lvl1, f"p={p1:.0e}")  # Leaky CV vs Gen error
annotate(1.1, 1.9, lvl1, f"p={p3:.1f}")  # Gen error vs CV
annotate(0.0, 2.0, lvl2, f"p={p2:.0e}")  # Leaky CV vs CV

ax.set_ylim(top=lvl2 + 0.09 * span)

fig.savefig("gpfa_decoding_diff.svg", transparent=True)